# B3 · Seen, not touched

Places around Bright Angel that Perseverance **saw** from its stops but never **drove to**.
Method and limits: `viewshed.py` docstring and `docs/validation.md`. Checks: `python -m pytest notebooks/test_viewshed.py`.

**Inputs** (from the assets bundle, folder set by `ARIADNE_ASSETS`, default `~/ariadne-assets/bright-angel/`):
`dem_1m.tif`, `waypoints.geojson`, `track.geojson`, optional `cost_to_home_2m.f32`.
Without them the notebook runs on a **SYNTHETIC** terrain and writes to `data/b3_synthetic/` (git-ignored), never to `notebooks/out/`.

**Target rule (option A, chosen by Bunyamin):** only ground seen from within `CLOSE_M` = 150 m counts (Mastcam-Z resolves ~5 cm there), cut into `TILE_M` = 50 m tiles; a tile with at least 100 m² of that ground is a candidate. Both numbers are Assumed.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

NB = Path.cwd() if (Path.cwd() / "viewshed.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NB))
import viewshed as vs

aoi = vs.load_aoi()
origin = vs.projected_origin(aoi)
height = vs.observer_height()
ASSETS = Path(os.environ.get("ARIADNE_ASSETS", Path.home() / "ariadne-assets" / "bright-angel")).expanduser()
need = ["dem_1m.tif", "waypoints.geojson", "track.geojson"]
REAL = all((ASSETS / f).exists() for f in need)
OUT = vs.OUT if REAL else vs.REPO / "data" / "b3_synthetic"
print("real data" if REAL else f"SYNTHETIC mode: {ASSETS} lacks {[f for f in need if not (ASSETS / f).exists()]}")
print("observer height", height, "m; max distance", vs.MAX_DISTANCE_M, "m; close view", vs.CLOSE_M, "m; tile", vs.TILE_M, "m; touched radius", vs.TOUCH_RADIUS_M, "m")

## Inputs

In [ ]:
if REAL:
    grid = vs.read_dem(ASSETS / "dem_1m.tif", origin)
    waypoints = vs.read_points(ASSETS / "waypoints.geojson")
    track = vs.read_lines(ASSETS / "track.geojson")
    source = ("Viewsheds (gdal_viewshed, observer 2.0 m, 600 m) from Perseverance waypoints on the HiRISE 1 m DTM; "
              "ground seen from within 150 m, minus 10 m around the rover track, in 50 m tiles; notebooks/viewshed.ipynb (task B3)")
else:
    # SYNTHETIC: 600 x 500 m at 1 m, rolling ground, one ridge and one mound; three made-up stops on a made-up track
    import rasterio
    from rasterio.transform import from_origin
    W, H = 600, 500
    yy, xx = np.mgrid[0:H, 0:W].astype("float32")
    z = 3 * np.sin(xx / 70) + 2 * np.cos(yy / 55)
    z += 14 * np.exp(-((xx - 330) / 12) ** 2) * (yy > 120)          # ridge, open at the south end
    z += 9 * np.exp(-(((xx - 480) ** 2 + (yy - 120) ** 2) / 40 ** 2))  # mound
    syn = OUT / "dem_synthetic.tif"; OUT.mkdir(parents=True, exist_ok=True)
    with rasterio.open(syn, "w", driver="GTiff", width=W, height=H, count=1, dtype="float32",
                       crs=rasterio.crs.CRS.from_proj4(aoi["crs"]["proj4"]),
                       transform=from_origin(origin[0], origin[1] + H, 1, 1)) as ds:
        ds.write(z.astype("float32"), 1)
    grid = vs.read_dem(syn, origin)
    waypoints = [{"x": 80.5, "y": 250.5, "sol": 1, "rmc": "syn_1"}, {"x": 200.5, "y": 300.5, "sol": 2, "rmc": "syn_2"},
                 {"x": 260.5, "y": 420.5, "sol": 3, "rmc": "syn_3"}]
    track = [np.array([[w["x"], w["y"]] for w in waypoints])]
    source = "SYNTHETIC terrain and stops for testing the method"
print("DEM", grid.shape, "px", grid.px, "m;", len(waypoints), "waypoints;", len(track), "track lines")

## Viewsheds → seen, not touched → targets

In [ ]:
res = vs.seen_not_touched(grid, waypoints, track, height)
cth = None
if REAL and (ASSETS / "cost_to_home_2m.f32").exists():
    W2, H2 = int(aoi["size_m"][0] / 2), int(aoi["size_m"][1] / 2)
    cth = np.fromfile(ASSETS / "cost_to_home_2m.f32", dtype="<f4").reshape(H2, W2)
vs.reachable(res.targets, cth, 2.0, aoi["size_m"][1])
fc = vs.write_outputs(res, OUT, source=source, synthetic=not REAL)
seen = (res.count > 0).mean()
print(f"{len(res.used)} viewsheds; seen at least once: {seen:.1%} of the DEM; touched: {res.touched.mean():.1%}; "
      f"seen from within {vs.CLOSE_M:.0f} m: {(res.closest <= vs.CLOSE_M).mean():.1%}; candidate tiles: {len(res.targets)}")

## Figure for the project page

In [ ]:
from matplotlib.colors import ListedColormap
dz_dy, dz_dx = np.gradient(np.nan_to_num(grid.z), grid.px)
shade = np.clip(0.6 - 0.5 * dz_dx + 0.5 * dz_dy, 0, 1)
x_left, y_top = grid.west - origin[0], grid.north - origin[1]
ext = [x_left, x_left + grid.shape[1] * grid.px, y_top - grid.shape[0] * grid.px, y_top]
ext2 = [x_left, x_left + res.snt2.shape[1] * 2 * grid.px, y_top - res.snt2.shape[0] * 2 * grid.px, y_top]
fig, ax = plt.subplots(figsize=(9, 7.2))
ax.imshow(shade, cmap="gray", extent=ext, vmin=0, vmax=1)
ax.imshow(np.ma.masked_where(res.count == 0, np.ones(res.count.shape)), cmap=ListedColormap(["#3b82c4"]), alpha=0.25, extent=ext)
ax.imshow(np.ma.masked_where(~res.snt2, np.ones(res.snt2.shape)), cmap=ListedColormap(["#f28c28"]), alpha=0.8, extent=ext2)
for line in track:
    ax.plot(line[:, 0], line[:, 1], color="white", lw=1.2)
ax.scatter([w["x"] for w in res.used], [w["y"] for w in res.used], s=8, color="white", zorder=3)
for t in res.targets[:12]:
    ax.annotate(t["id"], (t["x"], t["y"]), color="black", fontsize=7, weight="bold",
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.8))
ax.set_title(("SYNTHETIC: " if not REAL else "") + "Seen from <150 m, never visited (orange); seen (blue); track (white)", fontsize=10)
ax.set_xlabel("x east (m from AOI south-west corner)"); ax.set_ylabel("y north (m)")
fig.tight_layout(); fig.savefig(OUT / "fig_seen_not_touched.png", dpi=150)

## Candidate targets (Bunyamin chooses 3–5)

Sorted: reachable first (finite `cost_to_home`), then seen by more waypoints, then closer. Claude only ranks.

In [ ]:
def key(t):
    c = t.get("cost_to_home_min")
    ok = c is not None and np.isfinite(c)
    return (not ok, -t["n_views"], t["distance_m"])
for t in sorted(res.targets, key=key)[:15]:
    c = t.get("cost_to_home_min")
    home = "unknown" if c is None else ("unreachable" if not np.isfinite(c) else f"{c:.0f} min home")
    print(f'{t["id"]:>4}  ({t["x"]:7.1f}, {t["y"]:7.1f})  {t["area_m2"]:8.0f} m2  {t["distance_m"]:6.0f} m from sol {t["sol"]}  seen by {t["n_views"]:2d} stops  {home}')